In [22]:
from pathlib import Path
import qlib
import pickle
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from qlib.data import D
from pipeline.utils import prints, init_log_file, get_last_trading_day, calculate_vwap, add_cross_sectional_features, g_safe_features, load_calibrations, lookup_calibration
from pipeline.portfolio_builder import build_long_short_portfolio
from pipeline.display_utils import print_safe_trades
from pipeline.features import compute_all_features, momentum_label
from pipeline.daily_logger import run_daily_logging
from pipeline.performance.daily_summary import run_daily_summary
from pipeline.performance.reason_attribution import run_reason_attribution
from pipeline.execution.intraday import simulate_execution_intraday

# Stability modules
from stability import run_feature_drift_monitor, run_rolling_ic_monitor, run_recent_ic_monitor


In [23]:
START_DATE = "2018-01-01"
END_DATE = (datetime.today() - timedelta(days=0)).strftime("%Y-%m-%d")
MODEL_PATH = "trained_model_2.pkl"
SAFE_FEATURES = g_safe_features()
SAFE_DF_PATH = Path("artifacts/safe_entries.parquet")
TOP_K_LONG = 20
TOP_K_SHORT = 20
IC_WINDOW_DAYS = 60

init_log_file("logs/top_long_short.log")


In [ ]:
import importlib


NameError: name 'pipeline' is not defined

In [26]:
# -----------------------------
# Init Qlib
# -----------------------------
qlib.init(provider_uri="C:/Users/harve/.qlib/qlib_data/us_data", region="us")

# Load instruments
instrument_path = r"C:/Users/harve/.qlib/qlib_data/us_data/instruments/all.txt"
with open(instrument_path, "r") as f:
    instruments = [line.strip().split("\t")[0] for line in f if line.strip()]

# -----------------------------
# Load model + training columns
# -----------------------------
with open(MODEL_PATH, "rb") as f:
    saved = pickle.load(f)

model = saved["model"]
model_cols = saved["columns"]

prints(f"Loaded model from {MODEL_PATH}")
prints(f"Model expects {len(model_cols)} features with columns: {model_cols}")

# -----------------------------
# Load features
# -----------------------------
features = D.features(
    instruments=instruments,
    fields=SAFE_FEATURES,
    start_time=START_DATE,
    end_time=END_DATE,
)
X = features.copy()

X = add_cross_sectional_features(X)
prints(f"Features loaded with shape: {X.shape} and columns: {X.columns.tolist()}")


# Fill remaining NaNs with 0 only if necessary
X = X.fillna(0)
# ============================================================
# Clean NaN/Inf
# ============================================================
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(0)

# ============================================================
# Align columns
# ============================================================
missing = set(model_cols) - set(X.columns)
if missing:
    raise ValueError(f"Missing required model columns: {missing}")

X = X.reindex(columns=model_cols)


# -----------------------------
# Predict scores
# -----------------------------
scores = model.predict(X)
df = X.copy()
df["score"] = scores

df = df.sort_index()

# === SIGNAL MOMENTUM CALCULATIONS & PRICE CRASH INDICATOR ===
df = compute_all_features(df)
# ---------------------
# Determine latest date
# ---------------------
# how many instruments should exist on a "complete" day
n_instruments = df.index.get_level_values('instrument').nunique()
# count how many rows per date (i.e., how many instruments each date has)
per_date_counts = df.index.get_level_values('datetime').value_counts()
# keep only dates where ALL instruments are present
complete_dates = per_date_counts[per_date_counts == n_instruments].index
# latest common date across all instruments
latest_date = complete_dates.max()

# your df_today slice
df_today = df.xs(latest_date, level='datetime').copy()
prints(f"Latest available date: {latest_date}")
df_today = df_today.reset_index()  # bring instrument + datetime into columns


[55636:MainThread](2026-02-27 12:19:40,593) INFO - qlib.Initialization - [config.py:452] - default_conf: client.
[55636:MainThread](2026-02-27 12:19:40,599) INFO - qlib.Initialization - [__init__.py:79] - qlib successfully initialized based on client settings.
[55636:MainThread](2026-02-27 12:19:40,601) INFO - qlib.Initialization - [__init__.py:81] - data_path={'__DEFAULT_FREQ': WindowsPath('C:/Users/harve/.qlib/qlib_data/us_data')}


Loaded model from trained_model_2.pkl
Model expects 64 features with columns: ['$open', '$high', '$low', '$close', '$vol_5d', '$vol_10d', '$vol_20d', '$vol_10_20', '$vol_20_60', '$vol_5_60', '$ret_5d', '$ret_10d', '$ret_20d', '$mom_60d', '$mom_5d_z', '$mom_20d_z', '$price_above_ma20', '$price_above_ma60', '$trend_5_20', '$range_ma5', '$range_ma20', '$volume_log', '$volume_shock', '$volume_z', '$volume_vol', '$rank_vol_5d', '$rank_vol_10d', '$rank_vol_20d', '$rank_mom_20d', '$rank_mom_60d', '$rank_intraday_range', '$rank_volume_log', '$days_since_ipo_cont', '$ipo_bucket', '$ret_5d_vol_scaled', '$trend_persist_ma20', '$trend_persist_ma60', '$vol_20d_resid_liq', '$ret_20d_vol_scaled', '$micro_imbalance_z_20', '$eps_actual_lag3', '$eps_est_lag3', '$eps_surprise_lag3', '$eps_ttm', '$eps_growth_yoy', '$surprise_std', '$surprise_pct', '$beat_streak', '$revision_trend', '$eps_momentum', '$earnings_yield', '$ret_5d_rank_xs', '$ret_10d_rank_xs', '$ret_20d_rank_xs', '$vol_10_20_rank_xs', '$ret_20

In [32]:
import importlib
import pipeline.utils as utils
importlib.reload(utils)
import pipeline.portfolio_builder as portfolio_builder
importlib.reload(portfolio_builder)


<module 'pipeline.portfolio_builder' from 'c:\\ws\\qlib\\pipeline\\portfolio_builder.py'>

In [33]:
df_today = utils.calculate_vwap(df_today)

# =======================================
# SAVE TODAY'S PREDICTIONS FOR ROLLING IC
# =======================================
df_pred_today = pd.DataFrame({
    "date": pd.to_datetime(latest_date),
    "symbol": df_today["instrument"],
    "pred": df_today["score"],
})

# Save daily predictions
pred_dir = Path("stability_outputs/daily_predictions")
pred_dir.mkdir(parents=True, exist_ok=True)
df_pred_today.to_csv(pred_dir / f"preds_{latest_date.date()}.csv", index=False)

# --------------------------
# Build long/short portfolio
# --------------------------
portfolio = portfolio_builder.build_long_short_portfolio(
    df_today=df_today,
    df_full=df,
    latest_date=latest_date,
    top_k_long=TOP_K_LONG,
    top_k_short=TOP_K_SHORT,
    momentum_label_fn=momentum_label,
)


KeyError: 'Column not found: close'